# Complete RISK INTELLIGENCE Pipeline

---

# 1. Setting Up Model

In [1]:
!pip install pymupdf

In [ ]:
!pip install peft==0.13.2

In [3]:
import os
import re
import json
import fitz

import torch
import torch.nn as nn

import pandas as pd
import joblib

from transformers import AutoTokenizer, DistilBertModel
from peft import PeftModel

In [23]:
CLASSIFY_MODEL_PATH = '/content/drive/MyDrive/MIRA/models/mira-distilbert-lora'

## Classifier

In [5]:
class MIRAClassifier(nn.Module):

    def __init__(
        self,
        num_issue,
        num_category,
        num_severity,
        num_recurring
    ):

        super().__init__()

        self.distilbert = DistilBertModel.from_pretrained(
            'distilbert-base-uncased'
        )

        hidden_size = self.distilbert.config.hidden_size

        self.issue_classifier = nn.Linear(
            hidden_size,
            num_issue
        )

        self.category_classifier = nn.Linear(
            hidden_size,
            num_category
        )

        self.severity_classifier = nn.Linear(
            hidden_size,
            num_severity
        )

        self.recurring_classifier = nn.Linear(
            hidden_size,
            num_recurring
        )

    def forward(self, input_ids, attention_mask):

        outputs = self.distilbert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        pooled_output = outputs.last_hidden_state[:, 0]

        issue_logits = self.issue_classifier(
            pooled_output
        )

        category_logits = self.category_classifier(
            pooled_output
        )

        severity_logits = self.severity_classifier(
            pooled_output
        )

        recurring_logits = self.recurring_classifier(
            pooled_output
        )

        return {
            'issue': issue_logits,
            'category': category_logits,
            'severity': severity_logits,
            'recurring': recurring_logits
        }

### Load Label Mappings

In [6]:
with open(
    os.path.join(
        CLASSIFY_MODEL_PATH,
        'label_mappings.json'
    ),
    'r'
) as f:

    label_mappings = json.load(f)

### Load Tokenizer

In [7]:
tokenizer = AutoTokenizer.from_pretrained(
    'distilbert-base-uncased'
)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


## Model

In [8]:
model = MIRAClassifier(
    num_issue = len(label_mappings['issue']),
    num_category = len(label_mappings['category']),
    num_severity = len(label_mappings['severity']),
    num_recurring = len(label_mappings['recurring'])
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### Load LoRA Adapter

In [9]:
model.distilbert = PeftModel.from_pretrained(
    model.distilbert,
    CLASSIFY_MODEL_PATH
)

### Load Classification Heads

In [10]:
heads = torch.load(
    os.path.join(
        CLASSIFY_MODEL_PATH,
        'classification_heads.pth'
    ),
    map_location = 'cpu'
)

In [11]:
model.issue_classifier.load_state_dict(
    heads['issue_classifier']
)

model.category_classifier.load_state_dict(
    heads['category_classifier']
)

model.severity_classifier.load_state_dict(
    heads['severity_classifier']
)

model.recurring_classifier.load_state_dict(
    heads['recurring_classifier']
)

<All keys matched successfully>

### Set Device

In [12]:
device = torch.device(
    'cuda'
    if torch.cuda.is_available()
    else 'cpu'
)

model.to(device)

model.eval()

print("Device:", device)

Device: cuda


---

# 2. PDF Processing

In [13]:
PDF_PATH = '/content/drive/MyDrive/MIRA/Inspection Reports/Coal_Mine_Inspection_Report_Concise.pdf'

### Extract Text from PDF

In [18]:
def extract_pdf_text(pdf_path):

    document = fitz.open(
        pdf_path
    )

    text = ""

    for page in document:
        text += page.get_text()

    document.close()

    return text

In [15]:
def extract_findings(text):

    pattern = r'Finding\s+(F-\d+):\s*(.*?)(?=Finding\s+F-\d+:|(?:\n|\s)5\.\s*INSPECTOR[\'’]?S\s+REMARKS|$)'

    matches = re.findall(
        pattern,
        text,
        flags=re.DOTALL | re.IGNORECASE
    )

    findings = []

    for finding_id, finding_text in matches:

        findings.append({
            'finding_id': finding_id,
            'finding_text': finding_text.strip()
        })

    return findings

### Findings' Classification

In [16]:
def classify_finding(text):
    inputs = tokenizer(
        text, 
        return_tensors = 'pt',
        truncation = True,
        padding = True
    )

    inputs = {
        k: v.to(device)
        for k, v in inputs.items()
    }

    with torch.no_grad():
        outputs = model(
            input_ids = inputs['input_ids'],
            attention_mask = inputs['attention_mask']
        )

    issue_id = torch.argmax(
        outputs['issue'],
        dim=1
    ).item()

    category_id = torch.argmax(
        outputs['category'],
        dim=1
    ).item()

    severity_id = torch.argmax(
        outputs['severity'],
        dim=1
    ).item()

    recurring_id = torch.argmax(
        outputs['recurring'],
        dim=1
    ).item()

    return {
        'issue': label_mappings['issue'][issue_id],
        'category': label_mappings['category'][category_id],
        'severity': label_mappings['severity'][severity_id],
        'recurring': label_mappings['recurring'][recurring_id]
    }

In [17]:
pdf_text = extract_pdf_text(PDF_PATH)

### Extract Findings

In [19]:
findings = extract_findings(
    pdf_text
)

print(
    "Number of findings:",
    len(findings)
)

Number of findings: 6


## Classify Every Finding

In [20]:
results = []

for finding in findings:
    prediction = classify_finding(
        finding['finding_text']
    )

    results.append({
        'finding_id': finding['finding_id'],
        'finding_text': finding['finding_text'],
        'issue': prediction['issue'],
        'category': prediction['category'],
        'severity': prediction['severity'],
        'recurring': prediction['recurring']
    })

---

# 3. Risk Engine

In [24]:
import joblib

RISK_MODEL_PATH = '/content/drive/MyDrive/MIRA/models/mira-risk-classifier.pkl'
ENCODER_PATH = '/content/drive/MyDrive/MIRA/models/mira-risk-encoder.pkl'

risk_model = joblib.load(RISK_MODEL_PATH)
risk_encoder = joblib.load(ENCODER_PATH)

In [27]:
risk_results = []

for result in results:
    risk_input = pd.DataFrame([{
        'issue': result['issue'],
        'category': result['category'],
        'severity': result['severity'],
        'recurring': result['recurring']
    }])

    risk_input_encoded = risk_encoder.transform(risk_input)

    predicted_risk = risk_model.predict(risk_input_encoded)[0]

    probabilities = risk_model.predict_proba(risk_input_encoded)[0]

    predicted_index = list(risk_model.classes_).index(predicted_risk)

    risk_confidence = (
        probabilities[predicted_index] * 100
    )

    risk_results.append({
        'finding_id': result['finding_id'],
        'finding_text': result['finding_text'],
        'issue': result['issue'],
        'category': result['category'],
        'severity': result['severity'],
        'recurring': result['recurring'],
        'risk_confidence': round(
            risk_confidence,
            2
        )
    })


In [28]:
risk_results_df = pd.DataFrame(risk_results)

risk_results_df

,finding_id,finding_text,issue,category,severity,recurring,risk_confidence
0,F-01,During inspection of the eastern working panel...,Ventilation,Mine Safety,Medium,Yes,92.02
1,F-02,Examination of the roof support system in the ...,Roof Support,Mine Safety,Medium,Yes,93.56
2,F-03,The Load-Haul-Dump (LHD) equipment in the unde...,Emergency Preparedness,Mine Safety,Medium,Yes,89.96
3,F-04,The primary electrical substation serving unde...,Electrical Safety,Mine Safety,Medium,Yes,93.30
4,F-05,The main haul road surface exhibits significan...,Dust Suppression,Mine Safety,Medium,Yes,92.55
5,F-06,Emergency preparedness infrastructure was exam...,Emergency Preparedness,Emergency Safety,Low,Yes,90.67
